# 03 — Feature Engineering

Build all 96 features across 6 categories:
1. **Geography & Logistics** — Travel distance, timezones, surface, altitude
2. **Roster & Absences** — DP quality, FIFA window impact, expansion flag
3. **Tactical (Rolling)** — Attack/defense/BTS at 3/5/10 match windows
4. **Context & Motivation** — H2H, streaks, rest, congestion, rivalry, PPG
5. **Betting Odds** — Implied probabilities, market confidence, odds movement
6. **Elo Ratings** — Custom MLS Elo with season regression

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from mls_predictor.data_loader import load_raw_data
from mls_predictor.elo import compute_elo_history
from mls_predictor.feature_engine import build_all_features, get_feature_columns

In [ ]:
%%time
# Full pipeline
df = load_raw_data()
df = compute_elo_history(df)
df_feat = build_all_features(df)
print(f"\nResult: {len(df_feat)} matches × {len(df_feat.columns)} columns")

In [ ]:
# Feature groups
feature_cols = get_feature_columns()
print(f"Prediction features: {len(feature_cols['prediction'])}")
print(f"All features (incl. raw odds): {len(feature_cols['all'])}")
print(f"\nTarget distribution:")
for target in ['target_1x2', 'target_ou25', 'target_ggng']:
    print(f"  {target}: {dict(df_feat[target].value_counts().sort_index())}")

In [ ]:
# Correlation heatmap for top features
pred_cols = [c for c in feature_cols['prediction'] if c in df_feat.columns]
corr = df_feat[pred_cols].corr()

# Top correlations with target_1x2
if 'target_1x2' in df_feat.columns:
    target_corr = df_feat[pred_cols + ['target_1x2']].corr()['target_1x2'].drop('target_1x2', errors='ignore')
    top_corr = target_corr.abs().sort_values(ascending=False).head(20)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#00c48c' if v > 0 else '#ff4757' for v in target_corr[top_corr.index]]
    target_corr[top_corr.index].plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Top 20 Features — Correlation with 1X2 Target', fontweight='bold')
    ax.set_xlabel('Pearson Correlation')
    ax.axvline(0, color='white', linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Geography features distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_feat['travel_distance_km'].hist(bins=30, ax=axes[0], color='#00d4ff', alpha=0.8)
axes[0].set_title('Travel Distance (km)')

df_feat['timezones_crossed'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#7b2ff7')
axes[1].set_title('Timezones Crossed')

df_feat['venue_is_turf'].value_counts().plot(kind='bar', ax=axes[2], color=['#00c48c', '#ff4757'])
axes[2].set_title('Venue Surface (0=Grass, 1=Turf)')

plt.tight_layout()
plt.show()

In [ ]:
# Missing values in features
missing = df_feat[pred_cols].isnull().sum()
missing_pct = missing / len(df_feat) * 100
has_missing = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(has_missing) > 0:
    print(f"{len(has_missing)} features have missing values:")
    for feat, pct in has_missing.items():
        print(f"  {feat}: {pct:.1f}%")
else:
    print('✅ No missing values in prediction features!')

In [ ]:
# Save processed dataset
output_path = os.path.join('..', 'data', 'processed', 'features.parquet')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_feat.to_parquet(output_path, index=False)
print(f'✅ Features saved to {output_path}')
print(f'   {len(df_feat)} rows × {len(df_feat.columns)} columns')